# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant JSON-LD schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their `@id`s.

Let's inspect what record sets and fields are available in this dataset. Note: In Croissant, a record set is where the actual tabular data is referenced, usually a CSV file with metadata.

In [ ]:
# List available record sets, their @ids, and their fields (columns) by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s) in this dataset.")

for rs in record_sets:
    print(f"\nRecord Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields (columns):")
    for f in rs.fields:
        print(f"    {f.name} (@id: {f.id})  | Type: {f.data_type}")

## 3. Data Extraction
Load record data from the main record set into a DataFrame for analysis. We reference all entities by their `@id` (as per [FAIR² Croissant metadata specs](https://mlcommons.org/croissant/) and this notebook's rules).

**Tip:** Identify field and record set `@id` values from the overview (previous cell output).

In [ ]:
# If only one record set is present, we select it. Otherwise, modify as appropriate.
record_set_ids = [rs.id for rs in record_sets]
print("Record set @id(s):", record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded {len(df)} records from record set {rs_id}.")
    print(f"Columns: {df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filter, normalize, and group.

Let's pick a representative numeric field, such as percentage coverage, peptide count, or molecular weight (refer to fields above by their `@id`). If working directly in the notebook, update `<numeric_field_id>` and `<group_field_id>` as found in your dataset (the example below assumes common proteomics fields).

In [ ]:
# Pick the main record set id
record_set_id = record_set_ids[0]  # Use first record set if only one present
df = dataframes[record_set_id]

# List numeric candidates; update according to your field @ids.
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print('Numeric fields in DataFrame:', numeric_candidates)

# For demonstration, let's try typical fields -- update as required.
numeric_field = None
for col in ['coverage_percent', 'coverage', 'percent_coverage', 'cr:coverage', 'peptide_count', 'cr:peptide_count', 'mol_weight', 'cr:mol_weight', 'cr:molecular_weight']:
    if col in df.columns:
        numeric_field = col
        break
# If none found, take first numeric:
if not numeric_field and numeric_candidates:
    numeric_field = numeric_candidates[0]
print(f"Using numeric field: {numeric_field}")

if numeric_field:
    threshold = df[numeric_field].mean()  # e.g., filter above mean by default
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a protein/annotation field
    group_field = None
    for field in ['cr:description', 'annotation', 'sample', 'cr:sample']:
        if field in df.columns:
            group_field = field
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the numeric field's distribution and explore relationships.

Let's plot a histogram of the filtered, normalized values and a boxplot grouped by the selected categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    # Boxplot by group field if present
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded FAIR² proteomics data using `mlcroissant`, referencing all entities and columns by their `@id`.
- Previewed available record sets and their associated fields.
- Loaded and filtered tabular protein data, normalized a key quantitative feature, and grouped by categorical annotation.
- Created distribution visualizations for exploratory analysis.

The FAIR² Croissant metadata enables transparent, reproducible data access workflows suitable for downstream ML applications and collaborative research.